# AI and LLMs with python
* lessons 21/22 taught "classical" machine learning with scikit-learn - models that predict a specific NUMBER or CATEGORY from structured, tabular features (rows and columns)
* this lesson is about a different, much newer kind of AI: **Large Language Models (LLMs)** - the "ChatGPT / Claude" kind of AI
* LLMs work on **unstructured, free-form TEXT** instead of neat tables - you throw a chunk of text at them and get text back
* we will learn the request/response SHAPE that basically every LLM provider uses, how to think about tokens/cost, some prompt engineering basics, and an honest look at the limits of these models
* this lesson reuses the exact local-sandbox-server pattern from lesson 14 (web and apis) - go back and skim that lesson again if any of the `http.server`/`threading`/`requests` code below looks unfamiliar

## what even is a LLM?
* a Large Language Model is a (really, really big) model trained on huge amounts of text from the internet, books, code, etc
* at its core, all it really does is: given some text so far, predict "what word/piece-of-text comes next" - over and over again, one piece at a time
* that sounds almost too simple to be useful, but it turns out that doing this VERY well, at VERY large scale, is powerful enough to hold conversations, write and explain code, summarize long documents, translate languages, and much more
* important contrast with lessons 21/22's classical ML:
  * classical ML (scikit-learn): structured INPUT (a table of features, e.g. `[age, income, num_purchases]`) -> a specific NUMBER or CATEGORY as output (e.g. "will churn: yes/no")
  * LLMs: unstructured, free-form TEXT as input -> free-form TEXT as output - there is no fixed set of "features", you just describe what you want in plain language
* under the hood LLMs are still a neural network (conceptually related to what lesson 22 mentioned about PyTorch/TensorFlow for training), just an enormous one trained specifically on text
* good news for us as python developers: we almost never need to train or even understand the internals of a LLM ourselves - we mostly just need to know how to correctly TALK to one over an API, which is what this lesson focuses on

## the "chat completion" API pattern
* nearly every LLM provider (OpenAI, Anthropic/Claude, and pretty much everyone else) settled on basically the same request/response SHAPE, usually called a **"chat completion"** api
* you send a list of `messages`, each one a small dict with two fields:
  * `role` - who is "speaking" this message, one of:
    * `"system"` - sets the overall behaviour/persona/rules for the assistant (e.g. "you are a helpful cooking assistant, always answer in bullet points")
    * `"user"` - the actual human input/question
    * `"assistant"` - the model's OWN previous replies
  * `content` - the actual text of that message
* you get back the model's next message (usually also `role: "assistant"`, with some `content` text)
* the important, slightly surprising bit: the model itself has **no memory** between separate API calls! a "conversation" is really just YOU resending the entire growing list of messages every single time, and the model reads through all of it again from scratch to figure out the context
* this is exactly why the message list keeps growing turn by turn - we will see this below with our own local mock server

## building our own tiny local mock chat api
* just like lesson 14, we build a tiny local server with `http.server` + `threading` so this lesson runs 100% reliably, fully offline, without needing any real (paid!) LLM api key
* our mock server accepts a POST request shaped exactly like a real chat completion request (`{"messages": [...]}`) and returns a response shaped exactly like a real one (`{"choices": [{"message": {"role": "assistant", "content": "..."}}]}`)
* the "brain" behind our mock is nothing fancy - just a python function doing simple keyword matching, picking one of a few canned replies - good enough to teach the request/response SHAPE, without pretending to be actual intelligence
* we NEVER send anything to a real LLM provider anywhere in this lesson - no api key needed, no network access needed, this whole thing works on a plane with wifi off

In [1]:
import json                                                     # to turn python dicts <-> JSON text, exactly like in lesson 14
import threading                                                 # to run our mock server in the background
from http.server import BaseHTTPRequestHandler, HTTPServer       # stdlib http server, zero extra installs needed
import requests                                                  # the third-party http CLIENT library, already used in lesson 14


# our super simple "brain" - looks at the user's message and picks a canned reply based on keyword matching
# this is deliberately NOT real AI, it is just enough to demonstrate the request/response SHAPE of a real LLM api
def generate_reply(user_text):
    text = user_text.lower()  # lowercase everything so our keyword matching isnt case sensitive
    if "hello" in text or "hi " in text or text.strip() == "hi":
        return "Hello! I am a very fake, rule-based stand-in for a real LLM."
    elif "weather" in text:
        return "No idea, sorry - I am just a canned-reply generator, not a real weather service (or a real AI)!"
    elif "bye" in text:
        return "Goodbye! (still just canned text here, no real intelligence happened at any point)"
    else:
        # default fallback: just echo the message back, so we always return SOMETHING sensible-looking
        return f"You said: '{user_text}'. No keyword matched, so I am just echoing it back."


# CANNED_REPLY_FN is a module-level "pluggable brain" - our handler below calls whatever function this currently points to
# this lets the exercises later swap in a smarter version WITHOUT needing to restart the server
CANNED_REPLY_FN = generate_reply


class MockLLMHandler(BaseHTTPRequestHandler):

    def do_POST(self):
        # real chat completion apis only really have one important route, something like "/v1/chat/completions"
        if self.path == "/v1/chat/completions":
            content_length = int(self.headers.get("Content-Length", 0))     # how many bytes of body to expect
            raw_body = self.rfile.read(content_length)                       # read exactly that many bytes
            request_data = json.loads(raw_body.decode("utf-8"))             # bytes -> string -> python dict

            messages = request_data.get("messages", [])   # this is the list of {"role":..., "content":...} dicts the client sent

            # a real model looks at the WHOLE conversation, but our tiny mock just reacts to the LAST "user" message
            last_user_message = ""
            for message in reversed(messages):             # walk backwards so we find the most recent one first
                if message.get("role") == "user":
                    last_user_message = message.get("content", "")
                    break

            reply_text = CANNED_REPLY_FN(last_user_message)   # ask our pluggable "brain" for a canned reply

            # this response shape mirrors a REAL chat completion response closely on purpose
            response_payload = {
                "choices": [
                    {"message": {"role": "assistant", "content": reply_text}}
                ]
            }
            body_bytes = json.dumps(response_payload).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body_bytes)))
            self.end_headers()
            self.wfile.write(body_bytes)
        else:
            # any other route -> we dont have that endpoint
            error_body = json.dumps({"error": f"no such route: {self.path}"}).encode("utf-8")
            self.send_response(404)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(error_body)))
            self.end_headers()
            self.wfile.write(error_body)

    def log_message(self, format, *args):
        pass  # silence the default per-request console logging, keeps our notebook output clean


# port 0 -> let the OS pick a free port for us, avoids hardcoding a port that might be taken
llm_server = HTTPServer(("127.0.0.1", 0), MockLLMHandler)
llm_port = llm_server.server_address[1]

# run it in a background daemon thread, so our notebook can act as the "client" at the same time
llm_thread = threading.Thread(target=llm_server.serve_forever, daemon=True)
llm_thread.start()

llm_base_url = f"http://127.0.0.1:{llm_port}"
print(f"mock LLM chat api running on {llm_base_url}")

mock LLM chat api running on http://127.0.0.1:37559


## having a "conversation" with our mock api
* remember: the model has no memory of its own, so a multi-turn conversation is just US keeping a growing `messages` list and resending the whole thing every time
* first turn: a `system` message (sets the assistant's persona) + one `user` message
* second turn: we take the SAME list, append the assistant's reply we got back, then append a NEW user message, and send the whole thing again - this is exactly how real chat apps implement "memory" of the conversation

In [2]:
# turn 1: start a fresh conversation with a system message + one user message
conversation = [
    {"role": "system", "content": "You are a helpful assistant for a python lesson. Keep replies short."},  # sets overall behaviour
    {"role": "user", "content": "hello there!"},                                                              # the human's first message
]

response = requests.post(f"{llm_base_url}/v1/chat/completions", json={"messages": conversation})  # send the whole message list as JSON
response.raise_for_status()                                                                          # like lesson 14 taught us, always check this
assistant_message = response.json()["choices"][0]["message"]                                         # dig into the response shape to get the reply
print("assistant:", assistant_message["content"])

# this is the important bit: we APPEND the assistant's reply to our growing history
# without this step the model (a real one too!) would have zero idea what was said before
conversation.append(assistant_message)
print("conversation so far has", len(conversation), "messages")

assistant: Hello! I am a very fake, rule-based stand-in for a real LLM.
conversation so far has 3 messages


In [3]:
# turn 2: add a NEW user message and resend the ENTIRE history (system + user + assistant + new user)
conversation.append({"role": "user", "content": "whats the weather like today?"})

response = requests.post(f"{llm_base_url}/v1/chat/completions", json={"messages": conversation})
response.raise_for_status()
assistant_message = response.json()["choices"][0]["message"]
print("assistant:", assistant_message["content"])

conversation.append(assistant_message)   # keep growing the history again, ready for a possible turn 3
print("conversation so far has", len(conversation), "messages")
print()
print("the full message history we are now carrying around (and would resend again on the next turn):")
for message in conversation:
    print(f"  [{message['role']}] {message['content']}")

assistant: No idea, sorry - I am just a canned-reply generator, not a real weather service (or a real AI)!
conversation so far has 5 messages

the full message history we are now carrying around (and would resend again on the next turn):
  [system] You are a helpful assistant for a python lesson. Keep replies short.
  [user] hello there!
  [assistant] Hello! I am a very fake, rule-based stand-in for a real LLM.
  [user] whats the weather like today?
  [assistant] No idea, sorry - I am just a canned-reply generator, not a real weather service (or a real AI)!


## tokens and cost
* real LLM providers dont price or limit usage by "characters" or even plain "words" - they use their own unit called a **token**
* a token is roughly "a chunk of text", often close to a word or a piece of a word (a "sub-word piece") - NOT exactly one character, and NOT exactly one word either
* both the text you SEND (input/"prompt" tokens) and the text you GET BACK (output/"completion" tokens) usually count towards cost and towards any max-length limit
* real tokenizers (like OpenAI's `tiktoken` or Anthropic's tokenizer) are fairly involved - they split text into sub-word pieces using a trained vocabulary, so "tokenization" might become something like `["token", "ization"]` - 2 tokens for what looks like 1 word
* we are NOT going to install a real tokenizer library here (this lesson stays dependency-free besides `requests`) - instead we use two common, very rough back-of-envelope rules of thumb, purely to build intuition

In [4]:
sample_text = "Large language models split text into tokens, not exactly words or characters."

# rough rule of thumb #1: roughly one token per word - simple, but ignores punctuation/sub-word splitting entirely
word_based_estimate = len(sample_text.split())

# rough rule of thumb #2: roughly one token per 4 characters - another very common approximation quoted in provider docs
char_based_estimate = len(sample_text) / 4

print("sample text:", sample_text)
print("word-based rough estimate:", word_based_estimate, "tokens")
print("char-based rough estimate:", round(char_based_estimate, 1), "tokens")

# BOTH of these are simplifications! a real tokenizer would give yet another (probably closer, but still different) number
# the point here is just: "token count roughly tracks text length", which is enough to reason about cost/limits at a gut-check level
# for anything where the EXACT number matters (billing, hitting a hard context limit) you would use the provider's real tokenizer library instead

sample text: Large language models split text into tokens, not exactly words or characters.
word-based rough estimate: 12 tokens
char-based rough estimate: 19.5 tokens


## prompt engineering basics
* "prompt engineering" is just a fancy name for "being clear and deliberate about what you ask the model for" - a few practical habits that consistently help:
* **be specific** - "summarize this in 3 bullet points for a 10 year old" works much better than a vague "summarize this"
* **give examples ("few-shot prompting")** - showing 1-2 example input/output pairs before your real question massively improves how well the model follows the pattern you want
* **ask for a specific output format** - e.g. "answer ONLY with valid JSON, using the field `translation`" makes the reply much easier to parse programmatically afterwards
* **use the `system` message for persona/rules** - things that should apply to the WHOLE conversation (tone, role, constraints, output format) belong in `system`, not repeated in every single `user` message
* **keep instructions and content clearly separated** - e.g. "Translate the following text: \"...\"" so the model cant confuse your instructions with the text it should act on
* **iterate** - if the first reply isnt what you wanted, refine the prompt and try again, the same way you would debug any other piece of code

In [5]:
# a well-structured example messages list for a "translate to french" task - showing several of the bullets above in one go
# NOTE: we are only building and printing this list here, we are NOT sending it anywhere (no real api call happens in this cell)
translation_prompt_messages = [
    {
        "role": "system",
        # the system message carries the PERSONA + the OUTPUT FORMAT rule, so we dont have to repeat it every turn
        "content": "You are a translation assistant. Always answer with valid JSON of the shape {\"translation\": \"...\"} and nothing else.",
    },
    {
        "role": "user",
        # a few-shot EXAMPLE first, so the model sees the exact pattern we want, then the REAL request
        "content": (
            'Example: Input: "good night" -> Output: {"translation": "bonne nuit"}\n'
            'Now translate this to French: "good morning"'
        ),
    },
]

for message in translation_prompt_messages:
    print(f"[{message['role']}]")
    print(message["content"])
    print()

[system]
You are a translation assistant. Always answer with valid JSON of the shape {"translation": "..."} and nothing else.

[user]
Example: Input: "good night" -> Output: {"translation": "bonne nuit"}
Now translate this to French: "good morning"



## what real code looks like (not executed here)
* everything above talked to our own local mock server - safe, free, offline, no api key needed
* real providers give you an official python SDK that wraps the exact same "chat completion" JSON shape we built by hand above, just with a nicer python object interface
* we are only showing this as a CODE EXAMPLE IN MARKDOWN below - it is intentionally never executed in this notebook, since it would need a real (paid) api key and real internet access to an external company's servers, which this offline educational lesson must never require or encourage

```python
# OpenAI-style SDK (pip install openai) - NOT installed or run in this lesson
from openai import OpenAI

client = OpenAI(api_key="sk-...")  # never hardcode a real key in committed code - use an environment variable instead!
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "hello there!"},
    ],
)
print(response.choices[0].message.content)  # same "choices[0].message.content" shape we used against our own mock!
```

```python
# Anthropic-style SDK (pip install anthropic) - NOT installed or run in this lesson
import anthropic

client = anthropic.Anthropic(api_key="sk-ant-...")  # again, load this from an environment variable/secret manager in real code
response = client.messages.create(
    model="claude-sonnet-4-5",
    system="You are a helpful assistant.",           # anthropic takes the system prompt as its own separate parameter
    messages=[{"role": "user", "content": "hello there!"}],
    max_tokens=100,
)
print(response.content[0].text)
```

* notice how closely both SDKs mirror the exact `messages` list of `{role, content}` dicts we built and sent by hand to our own mock server above - that is the whole point of this lesson: once you understand the underlying JSON shape, the "real" code is just a thin, convenient wrapper around it

## an honest caveat: LLMs can be confidently wrong
* LLMs predict "plausible-sounding next text" - this is NOT the same thing as "true text"
* this means a LLM can **hallucinate**: state something completely made up in a fluent, confident-sounding way, with zero indication that it might be wrong
* practical rules for building anything real with LLMs:
  * always verify important facts independently, especially anything safety, financial, medical or legal related - never blindly trust a generated answer for these
  * where possible, show the user WHERE an AI-generated answer came from (a source document, a search result, a citation) so they can check it themselves
  * treat LLM output as a fast DRAFT or a helpful starting point, not as a guaranteed-correct final answer
* this is one of the biggest practical differences to classical ML (lessons 21/22): a classical model outputs a number/category you can validate against test data with a clear accuracy metric - a LLM's free-form text answer is much harder to automatically verify as "correct" or "incorrect"

## a genuinely offline, deterministic "AI" example: lexicon-based sentiment
* before LLMs, "AI" in NLP (natural language processing) often meant much simpler, fully rule-based techniques - and they are still useful (fast, free, fully predictable) for simple cases today
* a classic example: a **lexicon-based sentiment scorer** - a small hardcoded dictionary of positive/negative words, count how many of each show up in a sentence, and use that to guess if the overall tone is positive/negative/neutral
* this technique predates modern LLMs by decades, needs no server, no network, no api key at all - just plain python, and it always runs the exact same way (deterministic), unlike a LLM which can phrase things differently every time

In [6]:
# a tiny, hardcoded lexicon - real-world sentiment lexicons have thousands of words, ours is deliberately small for teaching
POSITIVE_WORDS = {"good", "great", "love", "excellent", "happy", "awesome", "amazing"}
NEGATIVE_WORDS = {"bad", "terrible", "hate", "awful", "sad", "worst", "boring"}


def lexicon_sentiment(sentence):
    """Very simple rule-based sentiment guess: positive/negative/neutral, based on word counts."""
    words = sentence.lower().split()                                # naive tokenization: lowercase + split on whitespace
    cleaned_words = [word.strip(".,!?\"'") for word in words]        # strip common punctuation so "great!" still matches "great"

    positive_count = sum(1 for word in cleaned_words if word in POSITIVE_WORDS)  # count how many positive words we found
    negative_count = sum(1 for word in cleaned_words if word in NEGATIVE_WORDS)  # same for negative words

    if positive_count > negative_count:
        return "positive"
    elif negative_count > positive_count:
        return "negative"
    else:
        return "neutral"  # either a tie, or no lexicon word matched at all


# lets try it on a few example sentences
test_sentences = [
    "This movie was great, I loved the acting!",
    "This was a terrible and boring experience.",
    "This is a table.",  # neither positive nor negative word appears -> neutral
]

for sentence in test_sentences:
    print(f"{lexicon_sentiment(sentence):>8} <- {sentence!r}")

# no LLM, no server, no network call anywhere in this cell - fully deterministic, always gives the exact same result

positive <- 'This movie was great, I loved the acting!'
negative <- 'This was a terrible and boring experience.'
 neutral <- 'This is a table.'


In [7]:
# time to clean up the main mock LLM server, exactly like lesson 14 taught us
llm_server.shutdown()             # tells serve_forever() to stop its loop
llm_thread.join(timeout=2)         # wait for the background thread to actually finish
print("mock LLM server shut down, thread alive?", llm_thread.is_alive())

mock LLM server shut down, thread alive? False


## quick comparison to other languages
* the "chat completion" api is, underneath everything, just a plain REST/JSON HTTP api - exactly the kind lesson 14 already taught us to call with `requests`. that means ANY language capable of making a HTTP POST request (basically all of them) can call a real LLM api directly, even with no official SDK at all - our own mock server demo above proves the same point: it is just `requests.post(url, json=...)`
* nearly every provider also ships first-party or well-maintained SDKs in multiple languages on top of that same underlying protocol, so you rarely NEED to hand-roll the HTTP calls in practice
* JavaScript/Node.js arguably has the second most mature LLM tooling ecosystem after python - it is heavily used for chatbot frontends and serverless "AI" functions (e.g. a Vercel/AWS Lambda function calling a LLM api)
* higher-level frameworks like **LangChain** and **LlamaIndex** (for "chaining" multiple AI calls/tools together, retrieving documents, etc) offer both python AND JavaScript/TypeScript versions of essentially the same concepts - pick whichever fits the rest of your stack
* python remains the dominant language specifically for TRAINING or fine-tuning models yourself (recap lesson 22's mention of PyTorch/TensorFlow) - but simply CALLING an already-trained model's api, like we did the whole lesson, is realistically language-agnostic, it is just HTTP + JSON

## Exercises
1. extend our mock server with a new canned keyword-based response (e.g. reacting to the word "joke"), then have a short conversation with it that triggers your new reply
2. write a function `rough_token_estimate(text)` using the `len(text) / 4` rule of thumb, and test it on a couple of different sentences
3. write a well-structured example `messages` list (a `system` message + a `user` message) for a prompt asking an AI to translate a sentence to French - do NOT call any real api, just build and print the list
4. extend the lexicon-based sentiment scorer with 3 more words of your choice (positive and/or negative), and test it on a new sentence that uses at least one of them

In [8]:
# for the exercises we spin up a fresh little mock LLM server, so they also run independently/reliably
exercise_server = HTTPServer(("127.0.0.1", 0), MockLLMHandler)
exercise_port = exercise_server.server_address[1]
exercise_thread = threading.Thread(target=exercise_server.serve_forever, daemon=True)
exercise_thread.start()
exercise_base_url = f"http://127.0.0.1:{exercise_port}"
print("exercise mock LLM server running on", exercise_base_url)

# TODO: solve exercise 1 here (extend the mock server with a new keyword-based canned reply, then chat with it)


# TODO: solve exercise 2 here (rough_token_estimate function using len(text)/4)


# TODO: solve exercise 3 here (build+print a system+user messages list for a french translation prompt)


# TODO: solve exercise 4 here (extend the sentiment lexicon with 3 more words, test on a new sentence)

exercise mock LLM server running on http://127.0.0.1:41589


### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [9]:
# sample solution 1 - extend the mock server with a new canned keyword-based reply
def generate_reply_v2(user_text):
    text = user_text.lower()
    if "joke" in text:
        return "Why do programmers prefer dark mode? Because light attracts bugs! (a very canned joke)"
    else:
        return generate_reply(user_text)  # fall back to all the original keyword handling for everything else

# swap our mock server's "brain" - the server is already running, but do_POST looks up CANNED_REPLY_FN fresh on every request
CANNED_REPLY_FN = generate_reply_v2

# now have a short conversation that triggers our new canned reply
solution_conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "tell me a joke please"},
]
response = requests.post(f"{exercise_base_url}/v1/chat/completions", json={"messages": solution_conversation})
print("assistant:", response.json()["choices"][0]["message"]["content"])


# sample solution 2 - rough_token_estimate using the len(text)/4 rule of thumb
def rough_token_estimate(text):
    """Very rough token count estimate: roughly one token per 4 characters (a common simplification)."""
    return len(text) / 4

print("estimate for a short sentence:", rough_token_estimate("Hello there, how are you?"))
print("estimate for a longer sentence:", rough_token_estimate("This is a somewhat longer example sentence used purely for testing our rough token estimator."))


# sample solution 3 - a system+user messages list for a french translation prompt, no api call
french_translation_messages = [
    {"role": "system", "content": "You are a translation assistant. Only reply with the French translation, nothing else."},
    {"role": "user", "content": "Translate this sentence to French: 'The weather is nice today.'"},
]
for message in french_translation_messages:
    print(f"[{message['role']}] {message['content']}")


# sample solution 4 - extend the lexicon with 3 more words, then test it
POSITIVE_WORDS.update({"fantastic", "wonderful", "delicious"})   # 3 new positive words
NEGATIVE_WORDS.update({"disgusting", "disappointing", "annoying"})  # (feel free to pick different ones!)

print(lexicon_sentiment("The food was fantastic and absolutely delicious!"))
print(lexicon_sentiment("The service was disappointing and quite annoying."))

# clean up the exercise mock server too
exercise_server.shutdown()
exercise_thread.join(timeout=2)
print("exercise mock LLM server shut down, thread alive?", exercise_thread.is_alive())

assistant: Why do programmers prefer dark mode? Because light attracts bugs! (a very canned joke)
estimate for a short sentence: 6.25
estimate for a longer sentence: 23.25
[system] You are a translation assistant. Only reply with the French translation, nothing else.
[user] Translate this sentence to French: 'The weather is nice today.'
positive
negative


exercise mock LLM server shut down, thread alive? False


* congratulation you finished this lesson

## what we learned
* what a LLM is in plain words (predicting "what text comes next", trained on huge amounts of text) and how it differs from classical ML (lessons 21/22): free-form TEXT in/out, vs. structured features -> a specific number/category
* the "chat completion" api pattern nearly every LLM provider uses: a growing list of `messages`, each with a `role` (`system`/`user`/`assistant`) and `content`, and why the model has no memory of its own between calls
* building and talking to our own local, offline, mock chat api with `http.server` + `threading` + `requests`, safely reusing the exact pattern from lesson 14
* what a "token" roughly is, why providers price/limit by token count (input AND output), and two very rough back-of-envelope estimation rules of thumb (with their caveats)
* prompt engineering basics: being specific, few-shot examples, requesting a specific output format, and using the `system` message for persona/rules
* what real-world SDK code looks like for OpenAI-style and Anthropic-style chat completion apis (shown as reference only, never executed, since it needs a real paid api key)
* the honest limits of LLMs: they can hallucinate confidently-wrong answers, so important facts need independent verification and sources should be shown where possible
* a fully deterministic, offline, non-LLM "AI" technique that predates LLMs by decades: lexicon-based sentiment scoring
* that the underlying protocol behind every LLM api is just plain REST/JSON HTTP, so any language (and any SDK, or no SDK at all) can call it